# 04 — Baselines and enhanced rolling-origin evaluation

This notebook evaluates the same direct forecast targets that Notebook 05 will use from all five source tables. Every forecast has an explicit origin and only uses information available by that quarter. Results are reported overall and separately by table and forecast scope.


### What the setup code does and why

This code imports the data, plotting, and modelling tools; finds the project folders; loads the shared evaluation settings; and fixes the random seed. We need this so every later cell uses the same horizons, lag-window length, paths, and repeatable settings.

In [ ]:
# mount Google Drive and set the working directory to the project path
from google.colab import drive
drive.mount("/content/drive")
import os
os.chdir("/content/drive/MyDrive/JobAI")  # the project path
os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"

In [ ]:
import hashlib, json, math, os
from pathlib import Path
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import yaml
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def _find_repo():
    env = os.environ.get("JOBAI_REPO")
    if env:
        return Path(env).resolve()
    p = Path.cwd().resolve()
    for candidate in (p, *p.parents):
        if (candidate / "configs" / "eval.yaml").is_file():
            return candidate
    return p

REPO = _find_repo()
PRO = REPO / "data" / "processed"
REPORTS = REPO / "reports"
FIGURES = REPORTS / "figures"
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
CFG = yaml.safe_load((REPO / "configs" / "eval.yaml").read_text())
HORIZONS = [int(h) for h in CFG["horizons"]]
WINDOW = int(CFG["feature_window_quarters"])
SEED = int(CFG["seed"])
np.random.seed(SEED)
print("repo:", REPO)
print("horizons:", HORIZONS, "window:", WINDOW)

## Load all selected forecast targets

**What this code does:** reads the target catalog from Notebook 03 and loads every quality-accepted direct target from `11l1`, `11n1`, `12tu`, `12tw`, and non-duplicate `12r5` rows. Source family, scope, and measure remain attached to each series for later diagnostics.


### Summarize forecast accuracy

This code converts the individual prediction errors into MAE, RMSE, MASE, and sMAPE. It first measures every series separately and then averages across series, so a large or unusually complete series cannot dominate the panel result.

In [ ]:
selection_path = PRO / "selected_series.csv"
assert selection_path.is_file(), "Run notebook 03 first"
selection = pd.read_csv(selection_path)

def as_bool(series):
    return series.astype(str).str.lower().isin({"true", "1", "yes"})

selected_mask = as_bool(selection["selected"])
if "is_forecast_target" in selection:
    target_mask = as_bool(selection["is_forecast_target"])
else:
    target_mask = selection["role"].isin(["benchmark_target", "panel_target", "geographic_target"])
target_rows = selection[selected_mask & target_mask].copy()
assert not target_rows.empty, "No selected direct forecast targets found; rerun notebook 03"

def quarter_ordinal(q):
    return int(q[:4]) * 4 + int(q[-1]) - 1

series_data = {}
for table_id, selected_table in target_rows.groupby("table_id", sort=True):
    normalized_path = PRO / f"{table_id}__normalized.csv"
    assert normalized_path.is_file(), f"Missing normalized table: {normalized_path.name}"
    table_frame = pd.read_csv(normalized_path, low_memory=False)
    first_dims = json.loads(selected_table.iloc[0]["dimensions_json"])
    dimension_cols = list(first_dims)
    selected_lookup = {}
    metadata_lookup = {}
    for row in selected_table.itertuples(index=False):
        dims = json.loads(row.dimensions_json)
        signature = tuple(str(dims[column]) for column in dimension_cols)
        selected_lookup[signature] = row.series_id
        metadata_lookup[row.series_id] = {
            "series_family": row.series_family,
            "forecast_scope": row.forecast_scope,
            "measure_code": row.measure_code,
        }
    grouped = table_frame.groupby(dimension_cols, dropna=False, sort=False)
    for group_key, frame in grouped:
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        signature = tuple(str(value) for value in group_key)
        series_id = selected_lookup.get(signature)
        if series_id is None:
            continue
        frame = frame[["timeperiod_q", "value"]].copy()
        frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
        frame["quarter_index"] = frame["timeperiod_q"].map(quarter_ordinal)
        frame = frame.sort_values("quarter_index").reset_index(drop=True)
        assert frame["timeperiod_q"].is_unique, f"Duplicate quarter: {series_id}"
        assert np.diff(frame["quarter_index"]).tolist() == [1] * (len(frame) - 1), f"Target series has a time gap: {series_id}"
        series_data[series_id] = {"table_id": table_id, "frame": frame, **metadata_lookup[series_id]}
assert len(series_data) == len(target_rows), f"Loaded {len(series_data)} of {len(target_rows)} selected targets"
display(target_rows.groupby(["table_id", "forecast_scope"]).size().rename("n_series").reset_index())
print("direct forecast series loaded:", len(series_data))


## Expanding-window forecasts with feature parity

For a forecast made at origin `t`, Ridge training examples are restricted to windows whose targets are already known by `t`. The enhanced Ridge receives the same quarter, trend, volatility, and zero-pattern features produced in notebook 05. This creates a more feature-rich and fairer numerical baseline candidate for the enhanced LLM prompt.


### Save the baseline evidence

This code saves every prediction, the per-series metrics, and the overall summary. It also writes a manifest with checksums and experiment settings, giving later notebooks a permanent, traceable baseline report instead of relying on values displayed only in memory.

In [ ]:
splits = CFG["splits"]

def split_for_origin(q):
    if splits["val_start"] <= q <= splits["val_end"]:
        return "validation"
    if splits["test_start"] <= q <= splits["test_end"]:
        return "test"
    return None

def slope(values):
    values = np.asarray(values, dtype=float)
    return float(np.polyfit(np.arange(len(values), dtype=float), values, 1)[0]) if len(values) > 1 else 0.0

def enhanced_ridge_features(window_values, origin_quarter):
    values = np.asarray(window_values, dtype=float)
    recent = values[-4:]
    last = float(values[-1])
    scale = max(abs(last), float(np.std(values)), 1.0)
    normalized_window = ((values - last) / scale).tolist()
    engineered = [
        (values[-1] - values[-2]) / scale,
        (values[-1] - values[-5]) / scale,
        (np.mean(recent) - last) / scale,
        (np.mean(values) - last) / scale,
        slope(recent) / scale,
        slope(values) / scale,
        np.std(recent) / scale,
        np.std(values) / scale,
        np.mean(values == 0),
    ]
    quarter_one_hot = [1.0 if int(origin_quarter[-1]) == quarter else 0.0 for quarter in (1, 2, 3, 4)]
    return np.asarray([*normalized_window, *engineered, *quarter_one_hot], dtype=float)

prediction_rows = []
for series_id, info in series_data.items():
    frame = info["frame"]
    quarters = frame["timeperiod_q"].tolist()
    values = frame["value"].to_numpy(dtype=float)
    for origin_idx in range(WINDOW - 1, len(values)):
        origin = quarters[origin_idx]
        split = split_for_origin(origin)
        if split is None:
            continue
        history = values[:origin_idx + 1]
        seasonal_pairs = np.isfinite(history[4:]) & np.isfinite(history[:-4])
        seasonal_diffs = np.abs(history[4:][seasonal_pairs] - history[:-4][seasonal_pairs])
        mase_scale = float(np.mean(seasonal_diffs)) if len(seasonal_diffs) else np.nan
        if not np.isfinite(mase_scale) or mase_scale <= 0:
            continue
        for horizon in HORIZONS:
            target_idx = origin_idx + horizon
            if target_idx >= len(values):
                continue
            seasonal_idx = origin_idx + horizon - 4
            current_window = values[origin_idx - WINDOW + 1:origin_idx + 1]
            required_values = np.concatenate([current_window, [values[target_idx], values[seasonal_idx], values[origin_idx]]])
            if not np.isfinite(required_values).all():
                continue
            y_true = float(values[target_idx])
            forecasts = {
                "last_value": float(values[origin_idx]),
                "seasonal_naive": float(values[seasonal_idx]),
            }
            X_train, X_train_enhanced, y_train = [], [], []
            latest_train_target = None
            for train_origin in range(WINDOW - 1, origin_idx - horizon + 1):
                train_target = train_origin + horizon
                assert train_target <= origin_idx
                train_window = values[train_origin - WINDOW + 1:train_origin + 1]
                train_value = values[train_target]
                if not np.isfinite(train_window).all() or not np.isfinite(train_value):
                    continue
                X_train.append(train_window)
                X_train_enhanced.append(enhanced_ridge_features(train_window, quarters[train_origin]))
                y_train.append(train_value)
                latest_train_target = train_target
            if len(X_train) < 10:
                continue
            ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
            ridge.fit(np.asarray(X_train), np.asarray(y_train))
            forecasts["ridge"] = float(ridge.predict(current_window.reshape(1, -1))[0])
            ridge_enhanced = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
            ridge_enhanced.fit(np.asarray(X_train_enhanced), np.asarray(y_train))
            forecasts["ridge_enhanced"] = float(ridge_enhanced.predict(enhanced_ridge_features(current_window, origin).reshape(1, -1))[0])
            for model, y_pred in forecasts.items():
                abs_error = abs(y_true - y_pred)
                denom = (abs(y_true) + abs(y_pred)) / 2.0
                prediction_rows.append({
                    "table_id": info["table_id"], "series_family": info["series_family"],
                    "forecast_scope": info["forecast_scope"], "measure_code": info["measure_code"],
                    "series_id": series_id, "split": split,
                    "model": model, "horizon_q": horizon, "origin_quarter": origin,
                    "target_quarter": quarters[target_idx], "y_true": y_true, "y_pred": y_pred,
                    "error": y_true - y_pred, "abs_error": abs_error, "squared_error": (y_true - y_pred) ** 2,
                    "mase_scale": mase_scale, "scaled_abs_error": abs_error / mase_scale,
                    "smape_component_pct": 0.0 if denom == 0 else 100.0 * abs_error / denom,
                    "max_train_target_quarter": quarters[latest_train_target] if model.startswith("ridge") else origin,
                })
predictions = pd.DataFrame(prediction_rows)
assert not predictions.empty
assert not predictions.duplicated(["series_id", "split", "model", "horizon_q", "origin_quarter"]).any()
assert (predictions["target_quarter"].map(quarter_ordinal) > predictions["origin_quarter"].map(quarter_ordinal)).all()
assert (predictions["max_train_target_quarter"].map(quarter_ordinal) <= predictions["origin_quarter"].map(quarter_ordinal)).all()
print("prediction rows:", len(predictions))


### Compare test MAE visually

This code plots test MAE for each model and forecast horizon. It provides a quick comparison of which baseline makes the smallest average error, displays the chart inline, and saves a copy in the reports figures folder.

In [ ]:
def metric_summary(group):
    return pd.Series({
        "n_forecasts": len(group),
        "MAE": group["abs_error"].mean(),
        "RMSE": math.sqrt(group["squared_error"].mean()),
        "MASE": group["scaled_abs_error"].mean(),
        "sMAPE_pct": group["smape_component_pct"].mean(),
    })

series_keys = ["split", "table_id", "series_family", "forecast_scope", "measure_code", "series_id", "model", "horizon_q"]
by_series = (predictions.groupby(series_keys, sort=True)
             .apply(metric_summary, include_groups=False).reset_index())
summary_keys = ["split", "model", "horizon_q"]
summary = (by_series.groupby(summary_keys, sort=True)
           [["MAE", "RMSE", "MASE", "sMAPE_pct"]].mean().reset_index())
summary_counts = (by_series.groupby(summary_keys, sort=True)
                  .agg(n_forecasts=("n_forecasts", "sum"), n_series=("series_id", "nunique"))
                  .reset_index())
summary = summary.merge(summary_counts, on=summary_keys, how="left", validate="one_to_one")
scope_keys = ["split", "table_id", "forecast_scope", "model", "horizon_q"]
scope_summary = (by_series.groupby(scope_keys, sort=True)
                 .agg(MAE=("MAE", "mean"), RMSE=("RMSE", "mean"), MASE=("MASE", "mean"),
                      sMAPE_pct=("sMAPE_pct", "mean"), n_forecasts=("n_forecasts", "sum"),
                      n_series=("series_id", "nunique")).reset_index())
counts = predictions.groupby(["split", "model", "horizon_q"]).size().unstack(["model", "horizon_q"])
assert counts.notna().all().all(), "A baseline is missing from an evaluated split/horizon"
display(summary.round(3))
display(scope_summary[scope_summary["split"] == "test"].round(3))


In [ ]:
prediction_path = REPORTS / "baseline_predictions.csv"
series_metrics_path = REPORTS / "baseline_metrics_by_series.csv"
summary_path = REPORTS / "baselines.csv"
scope_summary_path = REPORTS / "baselines_by_scope.csv"
predictions.to_csv(prediction_path, index=False)
by_series.to_csv(series_metrics_path, index=False)
summary.to_csv(summary_path, index=False)
scope_summary.to_csv(scope_summary_path, index=False)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest = {
    "dataset": "All quality-accepted direct targets from the five-table forecast catalog",
    "series_count": len(series_data), "series_by_table": target_rows.groupby("table_id").size().astype(int).to_dict(),
    "horizons_q": HORIZONS, "feature_window_quarters": WINDOW,
    "models": ["seasonal_naive", "last_value", "ridge", "ridge_enhanced"],
    "engineered_feature_set": CFG.get("engineered_feature_set", "enhanced_v1"), "splits": CFG["splits"],
    "information_cutoff_assertion": "passed",
    "selection_catalog_sha256": sha256(selection_path),
    "normalized_inputs": {str((PRO / f"{tid}__normalized.csv").relative_to(REPO)): sha256(PRO / f"{tid}__normalized.csv") for tid in sorted(target_rows["table_id"].unique())},
    "outputs": {str(path.relative_to(REPO)): {"rows": len(frame), "sha256": sha256(path)} for path, frame in [
        (prediction_path, predictions), (series_metrics_path, by_series),
        (summary_path, summary), (scope_summary_path, scope_summary),
    ]},
}
manifest_path = REPORTS / "baseline_run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
for path in (prediction_path, series_metrics_path, summary_path, scope_summary_path, manifest_path):
    print("wrote:", path)


In [ ]:
test_mae = summary[summary["split"] == "test"].pivot(index="horizon_q", columns="model", values="MAE")
ax = test_mae.plot(kind="bar", figsize=(10, 5))
ax.set(title="Five-table rolling-origin test MAE", xlabel="Forecast horizon (quarters)", ylabel="Macro-average MAE")
ax.tick_params(axis="x", rotation=0)
ax.figure.tight_layout()
ax.figure.savefig(FIGURES / "04_test_mae_by_horizon.png", dpi=150, bbox_inches="tight")
plt.show()

### How to read this figure

Each group of bars is one forecast horizon: one, two, or four quarters ahead. Each colored bar is a baseline model, and a shorter bar means a lower test MAE and therefore a better average forecast. Compare models within the same horizon; do not compare bar heights across horizons as if they were the same forecasting task. This chart is a quick visual summary, while `baselines.csv` contains the complete MAE, RMSE, MASE, and sMAPE results.